In [ ]:
%load_ext autoreload
%autoreload 2
from seek_code import SEEK
from data_handler import EleHandler
from seek_code import SEEK
from concept_head import ConceptHead

from torch.utils.data import DataLoader, Subset
from torch.utils.data.sampler import RandomSampler

import torch
import torch.nn as nn
import torch.optim as optim

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [65]:
dataset = EleHandler( )

# We restrict ourselves to the EFA_IDI
train_indices, val_indices, test_indices = dataset.split_along_encounters(identification='EFA_IDI')
train_loader, val_loader, test_loader = [DataLoader(Subset(dataset, indices), batch_size=128, shuffle=(i==0), num_workers=4, pin_memory=True) for i, indices in enumerate([train_indices, val_indices, test_indices])]

# Verify the data in train_loader
#for preprocessed_image, subject_id, ele_id_label, identified, subject_SEEK_1hot, ele_SEEK_1hot, left_ear, right_ear, subject_SEEK, ele_SEEK in train_loader:
#    assert all(identified), "Train loader contains non-identified entries!"
#    print("All entries in the batch are identified.")


Filtered dictionary #season unique values: ['EFA_IDI']
Train indices: 3904, Validation indices: 729, Test indices: 715


/data/vision/beery/scratch/antoine/CBM_reid/methods/data_handler.py:139: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_groups, temp_groups = train_test_split(unique_groups, test_size=1-split_sizes[0], random_state=42)
/data/vision/beery/scratch/antoine/CBM_reid/methods/data_handler.py:142: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  # Get the indices for each group


In [50]:
ch = ConceptHead()
#ch.train(train_loader, val_loader)
ch.test(test_loader)


Test Loss: 0.0870
sex Accuracy: 93.89%
age Accuracy: 93.34%
right_tusk Accuracy: 76.02%
left_tusk Accuracy: 77.20%
R_tear_1 Accuracy: 53.66%
R_hole_1 Accuracy: 73.19%
R_tear_2 Accuracy: 66.05%
R_hole_2 Accuracy: 80.58%
L_tear_1 Accuracy: 50.85%
L_hole_1 Accuracy: 74.65%
L_tear_2 Accuracy: 65.25%
L_hole_2 Accuracy: 80.61%
right_extreme Accuracy: 89.03%
left_extreme Accuracy: 88.52%
ear_special Accuracy: 86.71%
body_special Accuracy: 89.59%
average Accuracy: 77.45%
whole_code Accuracy: 7.10%


(0.08699074635903041,
 {'sex': 0.9388715326786041,
  'age': 0.9334375063578287,
  'right_tusk': 0.7602430582046509,
  'left_tusk': 0.7719618082046509,
  'R_tear_1': 0.5366493066151937,
  'R_hole_1': 0.7318923672040304,
  'R_tear_2': 0.660503476858139,
  'R_hole_2': 0.8058159748713175,
  'L_tear_1': 0.5085416734218597,
  'L_hole_1': 0.7464583317438761,
  'L_tear_2': 0.652465283870697,
  'L_hole_2': 0.8061284720897675,
  'right_extreme': 0.89031250278155,
  'left_extreme': 0.8851736187934875,
  'ear_special': 0.8671006957689921,
  'body_special': 0.8959027826786041,
  'average': 0.7744661495089531,
  'whole_code': 0.07100694502393405})

In [59]:

# Would be nice to make this a method in the ConceptHead class called (freeze everything)
ch.layer.eval()
for param in ch.layer.parameters():
    param.requires_grad = False

ch.backbone.eval()
for param in ch.backbone.parameters():
    param.requires_grad = False

classifier_layer = nn.Sequential(
    nn.Linear(2304, 1017),
    nn.Sigmoid()
)

classifier_layer = classifier_layer.to('cuda')
criterion = nn.CrossEntropyLoss()  # For multiclass classification
optimizer = optim.Adam(classifier_layer.parameters(), lr=0.001)

In [60]:
def compute_accuracy(logits, labels):
    """Compute accuracy given logits and true labels."""
    _, predicted = torch.max(logits, dim=1)  # Get predicted classes
    correct = (predicted == labels).sum().item()
    total = labels.size(0)
    accuracy = correct / total
    return accuracy * 100  # Return percentage

In [66]:
def epoch_pass(loader, training = True):
    
    classifier_layer.train() if training else classifier_layer.eval()
    total_loss, total_accuracy, total_batches = 0,0,0

    for batch in loader:
        images, ele_id_label, subject_SEEK, left_ears, right_ears = batch[0].to('cuda'),  batch[2].to('cuda'), batch[4], batch[6].to('cuda'), batch[7].to('cuda')

        with torch.no_grad():
            embeddings = ch.backbone(images)
            if ch.crop_ears:
                left_embeddings = ch.backbone(left_ears)
                right_embeddings = ch.backbone(right_ears)
                embeddings = torch.cat((embeddings, left_embeddings, right_embeddings), dim=1)
        
        # Forward pass
        logits = classifier_layer(embeddings)
        
        # Loss with integer labels
        loss = criterion(logits, ele_id_label)  
        total_loss += loss.item()

        # Compute metrics
        total_accuracy += compute_accuracy(logits, ele_id_label)
        total_batches += 1

        # Backward pass and optimization
        if training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    # accumulate loss and accuracy
    epoch_loss = total_loss / len(loader)
    epoch_accuracy = total_accuracy / total_batches

    return epoch_loss, epoch_accuracy

In [ ]:
for epoch in range(100):
    
    # Reset the parameters of the classifier layer
    classifier_layer.apply(lambda m: m.reset_parameters() if hasattr(m, 'reset_parameters') else None)


    train_loss, train_accuracy = epoch_pass(train_loader, training=True)
    val_loss, val_accuracy = epoch_pass(val_loader, training=False)

    print(f"Epoch {epoch + 1} - Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}% - Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")

test_loss, test_accuracy = epoch_pass(test_loader, training=False)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")

Epoch 1/10 - Train Loss: 6.7792, Train Accuracy: 1.01% - Val Loss: 6.9862, Val Accuracy: 0.39%
Epoch 2/10 - Train Loss: 6.7129, Train Accuracy: 1.26% - Val Loss: 6.9907, Val Accuracy: 0.65%
Epoch 3/10 - Train Loss: 6.7072, Train Accuracy: 1.86% - Val Loss: 6.9927, Val Accuracy: 0.00%
Epoch 4/10 - Train Loss: 6.7069, Train Accuracy: 1.59% - Val Loss: 6.9981, Val Accuracy: 0.52%
Epoch 5/10 - Train Loss: 6.7076, Train Accuracy: 1.23% - Val Loss: 7.0019, Val Accuracy: 0.26%
Epoch 6/10 - Train Loss: 6.7092, Train Accuracy: 1.59% - Val Loss: 6.9893, Val Accuracy: 0.26%
Epoch 7/10 - Train Loss: 6.7085, Train Accuracy: 1.26% - Val Loss: 7.0292, Val Accuracy: 0.65%
Epoch 8/10 - Train Loss: 6.7073, Train Accuracy: 1.13% - Val Loss: 6.9968, Val Accuracy: 0.52%
Epoch 9/10 - Train Loss: 6.7037, Train Accuracy: 0.98% - Val Loss: 6.9919, Val Accuracy: 0.65%
Epoch 10/10 - Train Loss: 6.7089, Train Accuracy: 0.98% - Val Loss: 6.9839, Val Accuracy: 0.26%
Epoch 11/10 - Train Loss: 6.7005, Train Accuracy:

In [ ]:
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")

Test Loss: 6.2441, Test Accuracy: 21.88%


# Training a projector

In [29]:
# Freeze the classifier layer
classifier_layer.eval()
for param in classifier_layer.parameters():
    param.requires_grad = False

In [ ]:
def intervene(predicted_concepts, true_concepts, correction_probability = 0.5):
    for i in range(predicted_concepts.shape[0]):
        if torch.rand(1).item() < correction_probability:
            predicted_concepts[i] = true_concepts[i]
    # To-do correction
    return predicted_concepts


In [39]:
projector_1 = nn.Sequential(
    nn.Linear(63, 512),
    nn.LeakyReLU(),
    nn.Linear(512, 2304),
    nn.LeakyReLU()
)

projector_1 = projector_1.to('cuda')
criterion = nn.CrossEntropyLoss()  # For multiclass classification
optimizer = optim.Adam(projector_1.parameters(), lr=0.001)

In [ ]:
def epoch_pass(loader, training=True, intervention = 0):
    # Freeze Backbone
    ch.backbone.eval()
    for param in ch.backbone.parameters(): param.requires_grad = False

    #Freeze the concept head layer
    ch.layer.eval()
    for param in ch.layer.parameters(): param.requires_grad = False

    # Freeze the classifier layer
    classifier_layer.eval()
    for param in classifier_layer.parameters(): param.requires_grad = False

    # Are we training ?
    projector_1.train() if training else projector_1.eval()

    total_loss, total_accuracy, total_batches = 0, 0, 0

    for batch in loader:
        images, ele_id_label, subject_SEEK, left_ears, right_ears = batch[0].to('cuda'),  batch[2].to('cuda'), batch[4], batch[6].to('cuda'), batch[7].to('cuda')

        with torch.no_grad():
            embeddings = ch.backbone(images)
            if ch.crop_ears:
                left_embeddings = ch.backbone(left_ears)
                right_embeddings = ch.backbone(right_ears)
                embeddings = torch.cat((embeddings, left_embeddings, right_embeddings), dim=1)

            # Compute the concepts from the backbone embeddings
            concept_logits = ch.layer(embeddings)
            predicted_concepts = SEEK.closest_valid_one_hot(concept_logits)
            
            # Allow for intervention
            edited_concepts = intervene(predicted_concepts, subject_SEEK, intervention)
        
        # Projecting the concepts back to the embeddings space
        projected_concepts = projector_1(edited_concepts)
        # I want to do this > loss = criterion(projected_concepts, embeddings)
        
        # We add the concepts projected back to the intiial embeddings space
        edited_embeddings = projected_concepts + embeddings

        # Now we can pass the edited embeddings to the classifier
        logits = classifier_layer(edited_embeddings)

        # Compute the CE loss
        loss = criterion(logits, ele_id_label)

        # Backward pass and optimization
        if training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Compute metrics
        total_accuracy += compute_accuracy(logits, ele_id_label)
        total_batches += 1
        total_loss += loss.item()
    
    epoch_loss = total_loss / len(loader)
    epoch_accuracy = total_accuracy / total_batches

    return epoch_loss, epoch_accuracy
        


In [ ]:
for epoch in range(10):
    train_loss, train_accuracy = epoch_pass(filtered_train_loader, training=True, intervention = 0.0)
    val_loss, val_accuracy = epoch_pass(filtered_val_loader, training=False, intervention = 0.0 )

    print(f"Epoch {epoch + 1}/{10} - Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}% - Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")


test_loss, test_accuracy = epoch_pass(filtered_test_loader, training=False, intervention = 0.0)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")

Epoch 1/10 - Train Loss: 6.2352, Train Accuracy: 21.79% - Val Loss: 6.2163, Val Accuracy: 23.94%
Epoch 2/10 - Train Loss: 6.2346, Train Accuracy: 22.02% - Val Loss: 6.2143, Val Accuracy: 23.70%
Epoch 3/10 - Train Loss: 6.2324, Train Accuracy: 22.16% - Val Loss: 6.2139, Val Accuracy: 24.19%
Epoch 4/10 - Train Loss: 6.2307, Train Accuracy: 22.17% - Val Loss: 6.2121, Val Accuracy: 24.54%
Epoch 5/10 - Train Loss: 6.2285, Train Accuracy: 22.29% - Val Loss: 6.2108, Val Accuracy: 25.71%
Epoch 6/10 - Train Loss: 6.2269, Train Accuracy: 22.38% - Val Loss: 6.2091, Val Accuracy: 24.66%
Epoch 7/10 - Train Loss: 6.2260, Train Accuracy: 22.28% - Val Loss: 6.2085, Val Accuracy: 24.84%
Epoch 8/10 - Train Loss: 6.2245, Train Accuracy: 22.47% - Val Loss: 6.2074, Val Accuracy: 25.54%
Epoch 9/10 - Train Loss: 6.2236, Train Accuracy: 22.55% - Val Loss: 6.2040, Val Accuracy: 25.34%
Epoch 10/10 - Train Loss: 6.2209, Train Accuracy: 22.50% - Val Loss: 6.2044, Val Accuracy: 24.60%


In [ ]:
for epoch in range(10):
    train_loss, train_accuracy = epoch_pass(filtered_train_loader, training=True, intervention = 0.5)
    val_loss, val_accuracy = epoch_pass(filtered_val_loader, training=False, intervention = 0.5)

    print(f"Epoch {epoch + 1}/{10} - Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}% - Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")


test_loss, test_accuracy = epoch_pass(filtered_test_loader, training=False, intervention = 0.5)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")